In [1]:
!pip install -q transformers accelerate bitsandbytes fastapi uvicorn nest-asyncio

# cloudflared for tunnel
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

import nest_asyncio
nest_asyncio.apply()

print("Environment ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.9 MB/s eta 0:00:00
Environment ready.


In [2]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

print(f"Loading {MODEL_ID} with 4-bit quantization...")
start = time.time()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

elapsed = time.time() - start
print(f"Model loaded in {elapsed:.1f}s on {model.device}")
if torch.cuda.is_available():
    print(f"VRAM used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

# Warm-up
messages = [{"role": "user", "content": "اختبار"}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
_ = model.generate(**inputs, max_new_tokens=10)
print("Model ready.")

Loading Qwen/Qwen2.5-7B-Instruct with 4-bit quantization...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model loaded in 812.3s on cuda:0
VRAM used: 5.2 GB


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model ready.


In [6]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Moroccan Legal RAG - Qwen2.5-7B LLM Service", version="1.0.0")

# ============================================================
# Moroccan legal system prompt
# ============================================================
SYSTEM_PROMPT = (
    "انت مساعد قانوني مغربي متخصص. تجيب على الاسئلة القانونية بناء على "
    "النصوص القانونية المقدمة فقط. لا تخترع معلومات غير موجودة في النصوص. "
    "تذكر ارقام المواد التي استندت اليها في اجابتك. "
    "اذا لم تكن الاجابة موجودة في النصوص المقدمة فقل ذلك بوضوح.\n\n"
    "قدم اجابتك بهذا التنسيق:\n"
    "1. اجابة مختصرة (سطرين الى ثلاثة)\n"
    "2. شرح مفصل (فقرة الى فقرتين)\n"
    "3. مثال عملي (ان امكن)\n"
    "4. المواد القانونية المستند اليها"
)


class LLMRequest(BaseModel):
    prompt: str
    max_tokens: int = 1024
    temperature: float = 0.3
    top_p: float = 0.9


class LLMResponse(BaseModel):
    response: str
    tokens_generated: int
    time_ms: float


@app.get("/")
async def root():
    return {
        "service": "Moroccan Legal RAG - Qwen2.5-7B LLM",
        "model": MODEL_ID,
        "device": str(model.device),
    }


@app.get("/health")
async def health():
    return {
        "status": "ok",
        "model": MODEL_ID,
        "device": str(model.device),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    }


@app.post("/generate", response_model=LLMResponse)
async def generate(request: LLMRequest):
    start = time.time()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": request.prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    generated_ids = None
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=request.max_tokens,
            temperature=request.temperature,
            top_p=request.top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
        generated_ids = outputs[0][input_len:]

    response_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    elapsed = (time.time() - start) * 1000

    tokens_count = len(generated_ids) if generated_ids is not None else 0

    del inputs, outputs, generated_ids
    torch.cuda.empty_cache()

    return LLMResponse(
        response=response_text.strip(),
        tokens_generated=tokens_count,
        time_ms=round(elapsed, 2),
    )


print("FastAPI app ready.")

FastAPI app ready.


In [7]:
test_prompt = """
النصوص القانونية:
[المادة 42 من مدونة التجارة]: يجب على كل شخص ملزم بالتسجيل في السجل التجاري
ان يبين في فاتوراته ومراسلاته رقم التسجيل ومكانه في السجل التحليلي.

[المادة 2244 من مدونة التجارة]: يجب على الشركات التجارية ان تشير في تصريحات
تسجيلها الى الاسماء الشخصية والعائلية للشركاء.

السؤال: ما هي شروط التسجيل في السجل التجاري للشركات التجارية؟
"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_prompt},
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

start = time.time()
outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.3, top_p=0.9, do_sample=True)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(f"Generated in {time.time() - start:.1f}s")
print(response)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Generated in 29.7s
1. شروط التسجيل في السجل التجاري للشركات التجارية:

يجب على الشركات التجارية أن تلتزم ببعض الشروط لتسجيلها في السجل التجاري، والتي تتضمن تقديم تصريح تسجيل يشير إلى الأسماء الشخصية والعائلية للشركاء، كما يجب أن تبين في فاتوراتها ومراسلاتها رقم التسجيل ومكانه في السجل التحليلي.

2. شرح مفصل:

تقوم المادة 2244 من مدونة التجارة بتوضيح أهمية ذكر الأسماء الشخصية والعائلية للشركاء في تصريحات التسجيل، حيث يعتبر هذا الإجراء ضروريًا لتحديد المسؤولية القانونية لكل شريك. أما المادة 42 من نفس المدونة، فتوضح أن الشركات التجارية ملزمة بإظهار رقم التسجيل ومكانه في السجل التحليلي في جميع الفواتير والمراسلات. هذه الخطوات تهدف إلى حماية حقوق الأطراف المعنية وتيسير التعاملات التجارية.

3. مثال عملي:

إذا كانت شركة تجارية تسمى "شركة أ" تمتلكها شخصان، "أحمد" و"محمد"، فيجب أن يتم تسجيل الشركة باسم "أحمد محمد" وتقديم هذا الاسم في تصريح التسجيل. كما يجب أن تظهر فواتير الشركة "شركة أ" رقم التسجيل واسم الشركاء "أحمد محمد".

4. المواد القانونية المستند اليها:

- المادة 2244 من مدونة التجارة
- ا

In [ ]:
import subprocess
import threading
import uvicorn
import re

def start_tunnel():
    time.sleep(3)
    process = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in process.stdout:
        print(line.strip())
        match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
        if match:
            print()
            print("=" * 60)
            print("PUBLIC TUNNEL URL:")
            print(f"   {match.group(0)}")
            print("=" * 60)
            print()
            print("Add to .env:")
            print(f'  LLM_SERVICE_URL={match.group(0)}')
            print()
            print("Test:")
            print(f'  curl {match.group(0)}/health')
            print()

tunnel_thread = threading.Thread(target=start_tunnel, daemon=True)
tunnel_thread.start()

print("Starting Qwen2.5-7B LLM service...")
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

Starting Qwen2.5-7B LLM service...


INFO:     Started server process [1244]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


2026-05-28T15:03:01Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-28T15:03:01Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-28T15:03:06Z INF +--------------------------------------------------------------------------------------------+
2026-05-28T15:03:06Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-05-28T15:03:06Z INF |  https://retained-dat-intellectual-turned.trycloudflar